# User Mapping Analysis

Created On: 7/25/2025

Created By: Juli Hirt

Objective of this notebook: 
* remove all instances of grey terms
* ensure all terms are 1:1 with their cleaned instances
* add a column for term stem


## Imports

In [ ]:
import pandas as pd
import numpy as np 
import ast
import string

from nltk.stem import PorterStemmer
from IPython.display import display


stemmer = PorterStemmer()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Data Loads

In [ ]:
df = pd.read_excel('Data/Inbound/all_terms_final - w_team_edits.xlsx', sheet_name='Raw Data')
df.head(3)

In [ ]:
rm_df = pd.read_excel('Data/Inbound/all_terms_final - w_team_edits.xlsx', sheet_name='Removed Terms')
rm_df.head(3)

In [ ]:
cln_df = pd.DataFrame(columns=df.columns)
cln_df.insert(7, 'QX_X.CleanedTermStem', '', True)
cln_df.head(3)

In [ ]:
rmf_df = pd.DataFrame(columns=df.columns)
rmf_df.head(3)

## Cleaning

In [ ]:
#duplicate the base dataframe to avoid modifying the original
temp_df = df.copy()
temp_df.insert(7, 'QX_X.CleanedTermStem', '', True)

# remove all whitespace from the term columns
temp_df['QX_X.UserProvidedMoodTerm'] = temp_df['QX_X.UserProvidedMoodTerm'].str.strip()
temp_df['QX_X.CleanedMoodTerm'] = temp_df['QX_X.CleanedMoodTerm'].str.strip()

In [ ]:
# duplicate the base dataframe to avoid modifying the original
temp_rm_df = rm_df.copy()

#remove all whitespace from the term columns
temp_rm_df['QX_X.UserProvidedMoodTerm'] = temp_rm_df['QX_X.UserProvidedMoodTerm'].str.strip()   
temp_rm_df['QX_X.CleanedMoodTerm'] = temp_rm_df['QX_X.CleanedMoodTerm'].str.strip()

In [ ]:
# Initialize variables for term cleaning
dic_arr = []
dic_temp = {
    'QX_X.UserProvidedMoodTerm': '',
    'QX_X.CleanedMoodTerm': '',
}
removed_term_count = 0
processed_row = 0

In [ ]:
# set the mode for processing
mode = 'test'
#mode = 'prod'

In [ ]:
for idx, row in temp_df.iterrows():
    user_provided_mood_term = row['QX_X.UserProvidedMoodTerm']
    cleaned_mood_term = row['QX_X.CleanedMoodTerm']
    
    # check if the term is in the removed terms
    if user_provided_mood_term in rm_df['QX_X.UserProvidedMoodTerm'].values or cleaned_mood_term in rm_df['QX_X.CleanedMoodTerm'].values:
        #for testing only
        if mode == 'test':
            print(f"Skipping removed term: {user_provided_mood_term} | {cleaned_mood_term}")
        removed_term_count += 1
        rmf_df = pd.concat([rmf_df, temp_df.iloc[[idx]]], ignore_index=True)

        continue

    # if the term is not in the removed terms, process it
    # 1 - check if the term is in the dictionary
    matched = False
    for dic in dic_arr:
        if user_provided_mood_term == dic['QX_X.UserProvidedMoodTerm'] or user_provided_mood_term.lower() == dic['QX_X.UserProvidedMoodTerm'].lower():
            matched = True
            # 2 - if it is, use the cleaned term from the dictionary
            cleaned_mood_term = dic['QX_X.CleanedMoodTerm']
            break
    if not matched:
        # 3 - if it is not, stem the term
        dic_temp_clone = dic_temp.copy()
        dic_temp_clone['QX_X.UserProvidedMoodTerm'] = user_provided_mood_term
        dic_temp_clone['QX_X.CleanedMoodTerm'] = cleaned_mood_term
        dic_arr.append(dic_temp_clone)

    # 4 - stem the cleaned term
    cleaned_mood_term_stem = stemmer.stem(cleaned_mood_term)

    # 5 - updated the dataframe
    temp_df.loc[idx, 'QX_X.CleanedMoodTerm'] = cleaned_mood_term
    temp_df.loc[idx, 'QX_X.CleanedTermStem'] = cleaned_mood_term_stem

    # 6 - add the cleaned term to the cleaned dataframe
    cln_df = pd.concat([cln_df, temp_df.iloc[[idx]]], ignore_index=True)
    processed_row += 1
    

if mode == 'test':
    print(f"Total removed terms: {removed_term_count}")


In [ ]:
# must equal 1522
removed_term_count + processed_row 

In [ ]:
#remove extra columns
rmf_df.drop(columns=['QX_X.CleanedTermStem'], errors='ignore', inplace=True)

In [ ]:
rmf_df.head(3)

In [ ]:
cln_df.head(10)

## Export Data

In [ ]:
with pd.ExcelWriter('Data/Outbound/cleaned_terms.xlsx') as writer:
    cln_df.to_excel(writer, sheet_name='Cleaned Terms', index=False)
    rmf_df.to_excel(writer, sheet_name='Removed Terms', index=False)